# TP 06 | Une vibration forte suffit-elle à faire vérifier la presse ? (corrigé)

**Chapitre 4. Apprendre à partir de probabilités**

Chez Nervure, Camille reçoit un relevé de vibration après chaque cycle. La maintenance veut repérer une dérive persistante avant de déplacer un technicien. Deux historiques fictifs de la ligne A se terminent par le même signal fort. La décision doit-elle être identique ?

**Pour démarrer.** Ouvrez ce notebook dans VS Code avec le noyau du dépôt. Les deux séquences initiales sont fournies ici. Les CSV du dossier donnees contiennent ensuite quatre campagnes historiques et un relevé à expertiser. Savoir lire une probabilité conditionnelle suffit. Au TP 05, elle portait sur le défaut d’un lot. Ici elle porte sur l’état de la presse.

**À remettre.** Une courbe annotée pour les deux scénarios, les matrices estimées sur les archives et une note de maintenance qui compare les tickets, les épisodes manqués et les délais. Trois défis facultatifs prolongent l’enquête.

## 1. Distinguer état et observation

**Atelier 6.1 du support MLB.** Sections 1 à 4.

![Six lectures de vibration sur deux historiques fictifs](../figures/tp06-signaux.svg)

La vibration est observable. L’état mécanique (« stable » ou « en dérive ») est caché. Un capteur peut vibrer alors que la ligne est stable et rester silencieux malgré la dérive. Relevez la première différence entre les séquences. Quelle information perdrait une règle qui lirait uniquement le dernier cycle ?

Au TP 05, une ligne du tableau décrivait un lot et la question portait sur son défaut. Ici, une ligne décrit un instant et la question porte sur la presse. Imaginez deux relevés consécutifs de vibration forte. Si un état mécanique persiste plusieurs cycles, le premier relevé change déjà ce que vous croyez sur l’état au deuxième cycle. Gardez donc les observations dans l’ordre, même si vous savez compter combien sont fortes en tout.

In [ ]:
# Repérer les premières divergences dans les deux journaux de presse.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sequences = {"Pic isolé": [0, 0, 0, 0, 0, 1], "Vibrations répétées": [0, 0, 0, 1, 1, 1]}
journal = pd.DataFrame(sequences, index=range(1, 7)).rename_axis("cycle")
display(journal)

**Corrigé.** Les séquences divergent au cycle 4. Au cycle 6, la valeur 1 est commune. Une règle fondée sur ce seul bit déclencherait la même décision dans les deux cas.

## 2. Faire évoluer une probabilité entre deux lectures

Une **chaîne de Markov** décrit la transition entre états. On suppose ici que, connaissant l’état actuel, le prochain état dépend seulement de celui-ci. Le **modèle de Markov caché** (HMM) ajoute un second mécanisme : chaque état produit des observations avec certaines probabilités. Dans cet exemple, ces paramètres sont des hypothèses fournies pour le TP.

![Transitions entre états et probabilités de vibration selon l’état](../figures/tp06-matrices.svg)

Lisez chaque ligne de gauche à droite. Sur 100 cycles qui commencent en dérive, combien devraient rester en dérive au cycle suivant ? Combien de lectures fortes attendez-vous sur 100 observations en état stable ? Pourquoi ces deux pourcentages répondent-ils à deux questions différentes ?

In [ ]:
# Vérifier le sens des matrices et la conservation des probabilités.
transition = np.array([[0.95, 0.05], [0.15, 0.85]])
emission = np.array([[0.90, 0.10], [0.20, 0.80]])
initial = np.array([0.90, 0.10])  # État avant le premier cycle observé.
for matrice in (transition, emission):
    np.testing.assert_allclose(matrice.sum(axis=1), 1)
print("État prévu au premier cycle :", initial @ transition)

**Corrigé.** 85 cycles sur 100 restent en dérive. Sur 100 cycles stables, on attend 10 vibrations fortes. Le premier nombre décrit la persistance mécanique et le second la fiabilité du signal. Avant toute mesure, le risque prévu au premier cycle vaut 0,90 × 0,05 + 0,10 × 0,85 = 0,13.

## 3. Mettre à jour le risque avec une vibration

Le **filtrage bayésien** utilise uniquement les mesures déjà reçues. Appelez $q$ la probabilité prévue de dérive, $v$ la vibration lue, $D$ la dérive et $S$ l’état stable. Après la transition, multipliez chaque probabilité par la vraisemblance de la lecture dans cet état. Divisez ensuite le poids de la dérive par la somme des deux poids.

$$
P(D \mid v)=\frac{q\,P(v\mid D)}{q\,P(v\mid D)+(1-q)\,P(v\mid S)}.
$$

**À vous.** Avec le risque prévu de 0,13, calculez les deux poids si la première vibration est forte. Refaites le calcul pour une vibration normale. Quelle lecture réduit le risque initialement prévu ?

![Prévision, pondération par la vibration forte et normalisation](../figures/tp06-normaliser.svg)

La colonne du milieu porte sur les trajectoires compatibles avec une lecture forte. Ses deux barres totalisent 0,191, la probabilité de recevoir cette lecture. Une fois la lecture reçue, seules ces trajectoires restent possibles. La dernière paire de barres répartit toute la probabilité entre leurs deux états.

In [ ]:
# Comparer les deux observations possibles avec le même état prévu.
prevu = initial @ transition
for vibration in [0, 1]:
    poids = prevu * emission[:, vibration]
    # Normalisez les deux poids, puis lisez la composante de dérive.
    posterior = poids / poids.sum()
    print(vibration, poids, posterior[1])

**Corrigé.** Une vibration forte donne des poids stable et dérive de 0,087 et 0,104. Le risque passe à 54,45 %. Une lecture normale donne 0,783 et 0,026, soit 3,21 % de risque. Le dénominateur tient compte des deux explications possibles du signal.

## 4. Suivre la presse cycle par cycle

Le résultat du cycle précédent devient le point de départ du suivant. Avant de lancer le filtre, indiquez à quel cycle les deux courbes devraient commencer à diverger. La fonction ci-dessous enregistre séparément le risque prévu avant la vibration et le risque révisé après sa lecture.

In [ ]:
# Rejouer les cycles dans leur ordre d’arrivée, sans utiliser de lecture future.
def filtrer(lectures, transitions=transition, emissions=emission):
    etat = initial.copy()
    lignes = []
    for cycle, vibration in enumerate(lectures, start=1):
        prevu = etat @ transitions
        poids = prevu * emissions[:, vibration]
        etat = poids / poids.sum()
        lignes.append((cycle, vibration, prevu[1], etat[1]))
    return pd.DataFrame(
        lignes, columns=["cycle", "vibration", "avant", "après"]
    ).set_index("cycle")


suivis = {nom: filtrer(lectures) for nom, lectures in sequences.items()}
risques = pd.DataFrame({nom: table["après"] for nom, table in suivis.items()})
display(risques.round(3))

![Risque filtré des deux séries et seuil de vérification](../figures/tp06-filtrage.svg)

**Décider pour Camille.** Le seuil opérationnel fictif est de 75 %, inclus. Tracez les deux historiques à partir de votre tableau. Relevez la valeur finale et le premier franchissement pour chaque série. Camille doit-elle vérifier la presse dans les deux cas ? Expliquez ce que cette décision implique pour les lots déjà produits.

In [ ]:
# Tracer le risque et repérer le premier cycle qui justifie une vérification.
seuil = 0.75
ax = risques.plot(marker="o", ylabel="Probabilité de dérive", ylim=(0, 1))
ax.axhline(seuil, linestyle="--", color="black")
plt.show()
# Pour chaque série, sélectionnez les cycles où le risque atteint le seuil.
for nom, serie in risques.items():
    alertes = serie[serie >= seuil]
    print(
        nom, round(serie.iloc[-1], 3), alertes.index.min() if len(alertes) else "aucun"
    )

**Corrigé.** Au dernier cycle, le pic isolé donne 34,4 % de risque, contre 94,6 % après trois vibrations fortes. Seule la seconde séquence déclenche une vérification, dès le cycle 5. Le filtre signale un état plausible de la presse. Pour décider du contrôle d’un lot, Camille conserve la règle du TP 05. Pour évaluer ce filtre sur le terrain, il faudrait des états confirmés par la maintenance.

## 5. Préparer le journal de maintenance

**Atelier 6.2 du support MLB.** Sections 5 à 8.

![États expertisés et vibrations des deux premières campagnes](../figures/tp06-campagnes.svg)

Camille vous confie quatre campagnes fictives de la presse A. Lors de ces essais historiques, une expertise a établi l’état de chaque cycle a posteriori. Vous pouvez donc apprendre les paramètres par comptage. Sur la prochaine campagne, seul le capteur sera disponible au moment de décider.

**Contrôle de réception.** Chargez le journal, vérifiez que chaque couple (campagne, cycle) est unique et que les cycles sont consécutifs au sein de chaque campagne. Calculez le nombre de transitions observables. Pourquoi la dernière ligne de A01 et la première de A02 ne constituent-elles pas une transition observée ? Sur la figure, repérez une vibration forte sur une presse stable.

In [ ]:
# Ouvrir les archives synthétiques depuis le TP ou depuis son corrigé.
from pathlib import Path

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent
dossier = racine / "donnees"
archives = pd.read_csv(dossier / "tp06_historique_presse.csv").sort_values(
    ["campagne", "cycle"]
)
display(archives.groupby("campagne").cycle.agg(["min", "max", "count"]))
# Cherchez les doublons puis comparez cycle et cycle précédent par campagne.
assert not archives.duplicated(["campagne", "cycle"]).any()
pas = archives.groupby("campagne").cycle.diff().dropna()
assert pas.eq(1).all()
print(len(pas), "transitions observées")
display(archives.query("derive_confirmee == 0 and vibration == 1").head(1))

**Corrigé.** Chaque campagne compte 80 cycles et 79 paires consécutives, soit 320 observations et 316 transitions. Les campagnes sont des essais séparés. Relier artificiellement leurs extrémités introduirait trois transitions dont on ignore le déroulement.

## 6. Estimer les deux matrices à partir de l’historique

![Effectifs distincts pour les transitions et les émissions](../figures/tp06-comptages.svg)

Pour la transition, associez l’état courant à l’état suivant **dans la même campagne**. Pour l’émission, associez l’état courant à la vibration du **même cycle**. La première matrice compte des paires, la seconde des cycles. L’index 0 représente « stable » ou « normale », l’index 1 « dérive » ou « forte ».

**À vous.** Reconstituez une ligne de chaque tableau de la figure, puis divisez chaque effectif par la somme de sa ligne. Quelle est la probabilité estimée de rester en dérive ? Quelle est la probabilité de vibration forte sur ligne stable ? Pourquoi les dénominateurs diffèrent-ils ?

In [ ]:
# Aligner les états dans chaque campagne et compter les observations par classe.
etat = archives.derive_confirmee
suivant = archives.groupby("campagne").derive_confirmee.shift(-1)
comptes_transition = pd.crosstab(etat, suivant).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0
)
comptes_emission = pd.crosstab(etat, archives.vibration).reindex(
    index=[0, 1], columns=[0, 1], fill_value=0
)
display(comptes_transition, comptes_emission)
# div(..., axis=0) permet de diviser chaque ligne par son propre total.
for table in [comptes_transition, comptes_emission]:
    display(table.div(table.sum(axis=1), axis=0).round(3))
assert comptes_transition.to_numpy().sum() == 316
assert comptes_emission.to_numpy().sum() == 320

**Corrigé.** Parmi les 87 transitions qui partent de la dérive, 69 y restent, soit 79,3 %. Parmi les 232 lectures en état stable, 24 sont fortes, soit 10,3 %. Un dernier cycle a une vibration connue même lorsque son état suivant est inconnu. Ces fréquences restent des estimations sur quatre campagnes.

### Prévoir la longueur d’un épisode

Si la presse est en dérive et si la probabilité de le rester vaut $s$ à chaque cycle, la probabilité que la dérive dure encore au moins trois transitions vaut $s^3$. La durée moyenne d’un épisode, premier cycle inclus, vaut $1/(1-s)$ sous cette hypothèse de transition constante. Quelle durée obtenez-vous ? Pourquoi une intervention de maintenance programmée pourrait-elle invalider cette prévision ?

In [ ]:
# Préparer les matrices de référence pour la suite du parcours.
def frequences(table):
    totaux = table.sum(axis=1)
    assert totaux.gt(0).all()
    return table.div(totaux, axis=0).to_numpy()


transitions_estimees = frequences(comptes_transition)
emissions_estimees = frequences(comptes_emission)
persistance = transitions_estimees[1, 1]
# Calculez la survie sur trois transitions et la durée moyenne en cycles.
print(persistance**3, 1 / (1 - persistance))

**Corrigé.** On obtient 49,9 % de probabilité de rester en dérive pendant les trois transitions et 4.8 cycles par épisode en moyenne. Une intervention modifie les chances de retour en état stable. Il faudrait distinguer les cycles avant et après cette action.

## 7. Rejouer une campagne sans connaître le diagnostic

![Lectures fortes, seuil fixe et risque filtré sur 48 cycles](../figures/tp06-releve-48.svg)

Les matrices sont maintenant fixées. Conservez le seuil de 0,75 annoncé par Camille. Chargez uniquement les vibrations de la nouvelle campagne et calculez le risque dans l’ordre des cycles. Une notification part au **début d’une série de cycles en alerte**. Les cycles suivants de la même série ne génèrent aucun nouveau ticket. Un cycle sous le seuil réarme la notification.

**À vous.** Repérez la première vibration forte, la première notification du filtre et tous les cycles où un ticket s’ouvre. Vérifiez ensuite qu’en arrêtant le rejeu au cycle 16 vous retrouvez les mêmes seize premières probabilités. Que détecterait ce contrôle si un calcul utilisait des moyennes centrées sur des lectures futures ?

In [ ]:
# Calculer les scores en ligne, avant d’ouvrir le rapport de l’expert.
releve = pd.read_csv(dossier / "tp06_releve_presse.csv").set_index("cycle")
rejeu = filtrer(releve.vibration, transitions_estimees, emissions_estimees)
signal_fort = releve.vibration.eq(1)
alerte_filtre = rejeu["après"].ge(0.75)


def ouvertures(alerte):
    return alerte & ~alerte.shift(1, fill_value=False)


display(rejeu.head(16).round(3))
# Utilisez ouvertures et comparez un rejeu court avec le début du rejeu complet.
print("Premier signal :", signal_fort[signal_fort].index.min())
print("Tickets du filtre :", releve.index[ouvertures(alerte_filtre)].tolist())
court = filtrer(releve.vibration.iloc[:16], transitions_estimees, emissions_estimees)
np.testing.assert_allclose(court["après"], rejeu["après"].iloc[:16])

**Corrigé.** Le premier signal fort arrive au cycle 3. Les tickets du filtre s’ouvrent aux cycles 12, 16, 39, 43. Plusieurs cycles hauts peuvent appartenir au même ticket. Le préfixe reste identique car le filtre n’utilise que le passé et la lecture courante. Ce contrôle peut révéler une dépendance au futur sur cette séquence.

## 8. Auditer les alertes avec l’expertise différée

![Diagnostic de la presse comparé aux deux règles sur chaque cycle](../figures/tp06-audit-episodes.svg)

L’expert a rendu son rapport pour les 48 cycles. Comparez le filtre à la règle simple « alerter dès qu’une vibration est forte ». Pour chaque règle, comptez les cycles stables en alerte, les cycles en dérive sans alerte et les tickets ouverts. Un ticket peut s’étendre sur plusieurs cycles et un faux signal isolé peut ouvrir un ticket entier. Indiquez à Camille ce qu’elle gagne et ce qu’elle risque avec le filtre.

In [ ]:
# Rapprocher les décisions déjà prises et le diagnostic de chaque cycle.
expertise = pd.read_csv(dossier / "tp06_expertise_presse.csv").set_index("cycle")
pd.testing.assert_index_equal(expertise.index, releve.index)
derive = expertise.derive_confirmee.eq(1)
regles = {"Capteur": signal_fort, "Filtre": alerte_filtre}
# Combinez les masques avec & et ~ pour distinguer les deux types d’erreur.
for nom, alerte in regles.items():
    print(
        nom,
        "fausses alertes (cycles)",
        (alerte & ~derive).sum(),
        "cycles manqués",
        (~alerte & derive).sum(),
        "tickets",
        ouvertures(alerte).sum(),
    )

**Corrigé.** La règle capteur donne 6 cycles de fausse alerte, 6 cycles de dérive manqués et 9 tickets. Le filtre donne respectivement 2, 10 et 4. La baisse des déplacements potentiels s’accompagne de périodes de dérive moins bien couvertes. Ces tickets simulent des notifications, sans intervention sur la trajectoire observée.

### Décision | Un épisode entier peut-il passer inaperçu ?

Reprenez la bande « Expertise » de la figure précédente. Un **épisode** est une suite ininterrompue de cycles en dérive. Pour chaque épisode, calculez le délai entre son début et le premier cycle en alerte pendant cet épisode. Comptez zéro si la règle est déjà active à son début. Inscrivez « manqué » si elle reste silencieuse jusqu’à la fin. Un épisode manqué ne doit pas entrer dans une moyenne comme un délai nul.

**À remettre à Camille.** Rédigez une recommandation pour les épisodes longs et une réserve pour les épisodes courts. Citez le nombre de tickets et un délai observé. Proposez un contrôle sur une future campagne avant d’autoriser une généralisation.

In [ ]:
# Former les épisodes à partir du diagnostic uniquement pour l’audit rétrospectif.
debuts = derive & ~derive.shift(1, fill_value=False)
numero_episode = debuts.cumsum()
for numero, segment in derive[derive].groupby(numero_episode):
    cycles_episode = segment.index
    print(
        "Épisode", numero, "du cycle", cycles_episode.min(), "au", cycles_episode.max()
    )
    # Restreignez chaque règle aux cycles de cet épisode avant de chercher la première alerte.
    for nom, alerte in regles.items():
        actifs = cycles_episode[alerte.loc[cycles_episode].to_numpy()]
        delai = int(actifs[0] - cycles_episode[0]) if len(actifs) else "manqué"
        print(nom, delai)

**Corrigé.** Les épisodes couvrent les cycles 10 à 18, 26 à 28 et 37 à 45. Les délais du capteur sont 1, 2, 1. Ceux du filtre sont 2, manqué, 2. Camille peut essayer le filtre pour prioriser les vérifications persistantes, mais le court épisode mérite un autre signal si une détection rapide est impérative. Sur la prochaine campagne, conserver le seuil et suivre ensemble le nombre de tickets, les épisodes manqués et les délais.

### Entre deux relevés, quelle amplitude est plausible ?

![Vibration estimée et intervalle qui s’élargit entre les relevés](../figures/tp06-amplitude.svg)

La maintenance apporte une courte série fictive de mesures analogiques, en mm/s. Le relevé binaire du HMM servait à estimer un état caché. Ici, un processus gaussien estime une courbe continue. Son noyau relie davantage les cycles proches, avec une longueur caractéristique fixée à deux cycles. Le bruit supposé du capteur a un écart type de 0,05 mm/s. Repérez la plus grande interruption dans les mesures. Entre les cycles 3, 6 et 9, où demanderiez-vous un relevé supplémentaire ?

La bande représente la moyenne estimée plus ou moins 1,96 écart type pour le signal latent à chaque cycle. Elle dépend du noyau et du bruit choisis. Les mesures des deux côtés sont disponibles pour cette analyse rétrospective.

In [ ]:
# Interpoler une amplitude analogique entre deux relevés de maintenance.
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

cycles_mesures = np.array([1, 2, 3, 4, 9, 10, 11])[:, None]
amplitudes = np.array([1.1, 1.2, 1.35, 1.4, 1.6, 1.5, 1.45])
cycles_grille = np.linspace(1, 11, 101)[:, None]
# Centrer en mm/s ; alpha est la variance du bruit de mesure supposé.
niveau = amplitudes.mean()
gp = GaussianProcessRegressor(kernel=RBF(2.0), alpha=0.05**2, optimizer=None)
gp.fit(cycles_mesures, amplitudes - niveau)
moyenne, ecart = gp.predict(cycles_grille, return_std=True)
moyenne += niveau
for cycle in [3, 6, 9]:
    i = np.abs(cycles_grille[:, 0] - cycle).argmin()
    print(
        f"Cycle {cycle} : {moyenne[i]:.2f} mm/s ; demi-largeur à 95 % : {1.96 * ecart[i]:.2f} mm/s"
    )

**Corrigé.** Les demi-largeurs sont de 0,09, 0,86 et 0,10 mm/s aux cycles 3, 6 et 9. Au cycle 6, le capteur est loin de chaque relevé. La bande y est plus large qu’aux cycles 3 et 9, où une mesure est disponible. Une lecture supplémentaire au milieu du trou réduirait cette incertitude. L’amplitude et sa bande sont en mm/s. Pour déclencher la maintenance, il faudrait encore définir une règle ou revenir à la probabilité de dérive du HMM.

## Défi facultatif | Le capteur devient plus bruyant

![Le capteur devient plus bruyant](../figures/tp06-bruit.svg)

Gardez les transitions et la sensibilité. Portez la probabilité de vibration forte en état stable de 10 % à 30 %. Quelle ligne de la matrice faut-il aussi rééquilibrer ? Anticipez la direction du changement puis vérifiez le dernier risque et la décision.

In [ ]:
# Tester une hypothèse en conservant les paramètres du parcours principal.
bruit = emission.copy()
lectures = sequences["Vibrations répétées"]
bruit[0] = [0.70, 0.30]
sensibilite = filtrer(lectures, emissions=bruit)
display(sensibilite.round(3))

**Corrigé.** Le dernier risque passe de 94,6 % à 58,2 %. Le même signal devient plus courant sur une ligne saine et apporte moins de preuve en faveur de la dérive.

## Défi facultatif | La vibration revient à la normale

![La vibration revient à la normale](../figures/tp06-retour.svg)

Ajoutez trois mesures normales après les trois fortes. Repérez le premier cycle repassant sous le seuil. Pourquoi ce retour ne prouve-t-il pas une réparation ? Vérifiez aussi que les six premières probabilités restent identiques après prolongation.

In [ ]:
# Tester une hypothèse en conservant les paramètres du parcours principal.
historique = sequences["Vibrations répétées"]
prolongement = historique + [0, 0, 0]
suite = filtrer(prolongement)
np.testing.assert_allclose(suite["après"].iloc[:6], risques["Vibrations répétées"])
display(suite.tail(4))

**Corrigé.** Au cycle 7 le risque retombe à 48,1 %. Le filtre tient compte du silence et de la possibilité de retour à l’état stable. Seule une inspection mécanique confirme l’état réel. Les nouveaux relevés laissent intact le calcul effectué aux cycles précédents.

### Défi facultatif | Un paquet absent est-il une vibration normale ?

![Deux trajectoires pour une lecture normale ou manquante au cycle 3](../figures/tp06-paquet-manquant.svg)

Après deux vibrations fortes, la télémétrie perd le troisième paquet. Supposons que la perte réseau est indépendante de l’état de la presse. Au cycle manquant, faites la transition habituelle et conservez la prévision. Comparez ce risque avec celui obtenu si une vibration normale avait réellement été reçue. Pouvez-vous utiliser zéro pour coder les deux situations ? Que changerait une panne de capteur provoquée par la surchauffe ?

In [ ]:
# Distinguer absence de paquet et vibration normale après deux signaux forts.
dernier_risque = filtrer([1, 1]).iloc[-1]["après"]
etat_precedent = np.array([1 - dernier_risque, dernier_risque])
prevision = etat_precedent @ transition
# Comparez prevision[1] au risque filtré issu des lectures [1, 1, 0].
risque_absent = prevision[1]
risque_normal = filtrer([1, 1, 0]).iloc[-1]["après"]
print("Paquet absent :", risque_absent, "lecture normale :", risque_normal)
assert risque_absent > risque_normal

**Corrigé.** Au cycle 3, l’absence de paquet laisse le risque à 75,6 %, contre 40.8 % quand une vibration normale est effectivement reçue. Le zéro est une observation qui favorise l’état stable. Une panne liée à la surchauffe serait informative et nécessiterait de modéliser la probabilité de perdre une lecture dans chaque état.

### Passage au TP 07

Archivez la recommandation de maintenance séparément de la règle d’inspection des lots. Le prochain TP portera sur la cohérence des décisions entre traitement batch et streaming.

[Filtrage et modèles de Markov cachés, cours de l’université de Berkeley](https://inst.eecs.berkeley.edu/~cs188/textbook/hmms/hmm.html)